In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os

In [2]:
# load ratings
path = os.path.abspath(os.path.join((os.getcwd()),  os.pardir, "data\\recommendation\\rating.csv") )
ratings = pd.read_csv(path)
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [1]:
ratings.shape

NameError: name 'ratings' is not defined

In [4]:
ratings.groupby("userId")["movieId"].count().min()
#each user has rated atleast 20 movies

20

In [5]:
# limits
n_users = 5000
n_movies = 1000

top_users = ratings['userId'].value_counts().head(n_users).index.tolist() ## users who rated more number of movies
top_movies = ratings['movieId'].value_counts().head(n_movies).index.tolist() ## movies that received more number of ratings

In [6]:
filtered = ratings[ratings['userId'].isin(top_users) & ratings['movieId'].isin(top_movies)]
filtered.shape

(2353665, 4)

In [7]:
filtered.groupby("userId")["movieId"].count().sort_values(ascending=False)

userId
8405      999
88604     994
118205    988
118754    964
74142     952
         ... 
15452     118
24443     114
59931     104
65687      87
42229      35
Name: movieId, Length: 5000, dtype: int64

In [8]:
filtered[filtered["userId"] == 42229]["rating"].value_counts()

rating
5.0    11
3.5     7
4.0     6
2.0     4
4.5     3
3.0     2
1.5     1
2.5     1
Name: count, dtype: int64

In [9]:
ratings[ratings["userId"] == 8405]["rating"].value_counts()

rating
4.0    1531
3.0    1442
2.5    1131
3.5    1084
2.0     900
4.5     534
5.0     438
1.5     382
1.0      73
Name: count, dtype: int64

In [10]:
filtered.groupby("movieId")["userId"].count().sort_values(ascending=False)

movieId
2571    4732
356     4691
1270    4669
480     4668
296     4643
        ... 
207      811
637      772
276      770
79       629
140      581
Name: userId, Length: 1000, dtype: int64

In [11]:
user_movie_df = filtered.pivot_table(index="userId", columns="movieId", values="rating")
user_movie_matrix = user_movie_df.fillna(0)
user_movie_matrix

movieId,1,2,3,5,6,7,10,11,14,16,...,70286,71535,72998,73017,74458,78499,79132,80463,81591,81845
userId,,,,,,,,,,,,,,,,,,,,,
54,4.0,3.0,0.0,3.0,3.0,0.0,4.0,5.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58,5.0,0.0,0.0,0.0,4.5,0.0,0.0,4.5,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
91,4.0,3.5,3.0,0.0,0.0,2.5,4.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
116,3.0,2.0,2.0,0.0,1.5,0.0,2.0,2.0,0.0,3.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138325,5.0,3.0,0.0,0.0,4.5,0.0,3.5,0.0,0.0,4.5,...,4.5,4.5,3.5,0.0,4.0,5.0,4.5,4.5,4.5,5.0
138382,3.0,4.0,3.0,0.0,0.0,3.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138397,0.0,0.0,5.0,0.0,5.0,0.0,4.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## User based CF

In [12]:
user_sim = cosine_similarity(user_movie_matrix) # similarity_matrix[i][j] = similarity between user i and user j
user_sim_df = pd.DataFrame(user_sim, index=user_movie_df.index, columns=user_movie_df.index)
user_sim_df

userId,54,58,91,104,116,156,208,271,294,298,...,138211,138254,138270,138301,138307,138325,138382,138397,138406,138437
userId,,,,,,,,,,,,,,,,,,,,,
54,1.000000,0.547123,0.531749,0.449013,0.522661,0.662074,0.525756,0.360460,0.543421,0.654510,...,0.400627,0.493111,0.530702,0.414576,0.486886,0.479598,0.620054,0.572552,0.423226,0.363397
58,0.547123,1.000000,0.578699,0.538901,0.485607,0.616222,0.631064,0.359214,0.556434,0.514658,...,0.464117,0.521729,0.561207,0.492370,0.549470,0.621205,0.494220,0.529059,0.509357,0.464631
91,0.531749,0.578699,1.000000,0.469473,0.589310,0.604817,0.554775,0.512525,0.667053,0.560605,...,0.548978,0.587856,0.513551,0.530053,0.522647,0.613625,0.492813,0.574972,0.538413,0.497103
104,0.449013,0.538901,0.469473,1.000000,0.279076,0.476359,0.610900,0.207724,0.397437,0.286026,...,0.353751,0.267870,0.538606,0.393347,0.490816,0.611429,0.347663,0.331793,0.384992,0.327474
116,0.522661,0.485607,0.589310,0.279076,1.000000,0.587215,0.441212,0.521328,0.619177,0.642129,...,0.520769,0.653749,0.434284,0.503389,0.438368,0.497056,0.511029,0.565360,0.502869,0.503115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138325,0.479598,0.621205,0.613625,0.611429,0.497056,0.534240,0.719528,0.512025,0.624426,0.387323,...,0.519936,0.549642,0.550825,0.623471,0.630409,1.000000,0.408288,0.499713,0.613723,0.637496
138382,0.620054,0.494220,0.492813,0.347663,0.511029,0.642799,0.415615,0.376889,0.505980,0.662916,...,0.390604,0.525286,0.457369,0.346077,0.396641,0.408288,1.000000,0.507327,0.403541,0.294775
138397,0.572552,0.529059,0.574972,0.331793,0.565360,0.607327,0.431368,0.435229,0.581106,0.568498,...,0.357144,0.550453,0.435667,0.468376,0.402111,0.499713,0.507327,1.000000,0.449442,0.407179


In [13]:
def predict_rating_user_based(user_id, movie_id, k=10): ##movie_id = unrated movie of current user
    if movie_id not in user_movie_df.columns or user_id not in user_movie_df.index:
        return np.nan
    
    movie_ratings = user_movie_df[movie_id] # get the particular movie ratings from all users
    valid_users = movie_ratings.dropna().index # remove users who have not rated that particular movie
    ratings = movie_ratings[valid_users] # get the particular movie ratings only from valid users

    sims = user_sim_df.loc[user_id]  # selecting the particular user row from the cosine similarity matrix. this will have he similarity scores of current user vs other users
    sims = sims[valid_users] # filter the similarity scores only for those valid users who rated that movie

    if sims.empty:
        return np.nan

    top_k_users = sims.sort_values(ascending=False).head(k) # filter top K users with higher similarity scores
    top_k_ratings = ratings.loc[top_k_users.index] #filter top K ratings of that particular movie by top K similar users


    pred = np.dot(top_k_users.values, top_k_ratings.values) / np.sum(np.abs(top_k_users.values)) # ratings vs similarity weighed average
    return pred


In [14]:
def recommend_top_n_user_based(user_id, n=5, k=10):
    if user_id not in user_movie_df.index:
        return []


    rated = user_movie_df.loc[user_id].dropna().index # rated movie list of an user
    unrated = [movie for movie in user_movie_df.columns if movie not in rated] # unrated movie list of an user


    predictions = []
    for movie_id in unrated: # loop all unrated movies to get recommendation
        #print(f"movie id: {movie_id}")
        pred = predict_rating_user_based(user_id, movie_id, k)  # passing each unrated movie id for that particular user to get top K predictions
        if not np.isnan(pred):
            predictions.append((movie_id, pred))
        #break

    top_n = sorted(predictions, key=lambda x: x[1], reverse=True)[:n]
    return top_n


In [15]:
top_recs_user_based = recommend_top_n_user_based(user_id=42229, n=5, k=10) #k = filter K users with highest similarity score; #n = filter top n results

print("User-based recommendations:")
for movie_id, score in top_recs_user_based:
    print(f"Movie {movie_id} → predicted rating: {score:.2f}")


User-based recommendations:
Movie 318 → predicted rating: 4.65
Movie 6016 → predicted rating: 4.49
Movie 58559 → predicted rating: 4.40
Movie 2324 → predicted rating: 4.40
Movie 2858 → predicted rating: 4.39


## Item based CF

In [16]:
# Item-based similarity calculation
item_sim = cosine_similarity(user_movie_matrix.T)
item_sim_df = pd.DataFrame(item_sim, index=user_movie_df.columns, columns=user_movie_df.columns)
item_sim_df.head()

movieId,1,2,3,5,6,7,10,11,14,16,...,70286,71535,72998,73017,74458,78499,79132,80463,81591,81845
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.766482,0.543301,0.531587,0.703963,0.540227,0.742627,0.646873,0.358172,0.672434,...,0.528650,0.465902,0.544533,0.487044,0.464872,0.505247,0.540868,0.469308,0.442242,0.445726
2,0.766482,1.000000,0.519800,0.521413,0.591841,0.490483,0.681443,0.545834,0.273971,0.574333,...,0.484466,0.449009,0.506605,0.469642,0.439463,0.421740,0.499386,0.402579,0.391750,0.379017
3,0.543301,0.519800,1.000000,0.536689,0.445541,0.441930,0.507917,0.500387,0.251055,0.444221,...,0.229940,0.220094,0.247252,0.238470,0.202749,0.213028,0.237879,0.207058,0.184000,0.186749
5,0.531587,0.521413,0.536689,1.000000,0.395942,0.492047,0.473851,0.513764,0.235717,0.388996,...,0.227456,0.220086,0.264409,0.239578,0.209724,0.242553,0.251905,0.231345,0.206923,0.202518
6,0.703963,0.591841,0.445541,0.395942,1.000000,0.427751,0.685438,0.542029,0.385855,0.751148,...,0.481858,0.416750,0.454762,0.427353,0.435528,0.372837,0.483997,0.425386,0.399266,0.382268


In [17]:
def predict_rating_item_based(user_id, movie_id, k=10):
    if movie_id not in user_movie_df.columns or user_id not in user_movie_df.index:
        return np.nan


    sims = item_sim_df[movie_id]
    user_ratings = user_movie_df.loc[user_id]
    valid_items = user_ratings.dropna().index


    sims = sims[valid_items]
    ratings = user_ratings[valid_items]


    if sims.empty:
        return np.nan


    top_k_items = sims.sort_values(ascending=False).head(k)
    top_k_ratings = ratings.loc[top_k_items.index]


    pred = np.dot(top_k_items.values, top_k_ratings.values) / np.sum(np.abs(top_k_items.values))
    return pred


In [18]:
def recommend_top_n_item_based(user_id, n=5, k=10):
    if user_id not in user_movie_df.index:
        return []


    rated = user_movie_df.loc[user_id].dropna().index
    unrated = [movie for movie in user_movie_df.columns if movie not in rated]


    predictions = []
    for movie_id in unrated:
        pred = predict_rating_item_based(user_id, movie_id, k)
        if not np.isnan(pred):
            predictions.append((movie_id, pred))


    top_n = sorted(predictions, key=lambda x: x[1], reverse=True)[:n]
    return top_n


top_recs_item_based = recommend_top_n_item_based(user_id=42229, n=5, k=10)

print("\nItem-based recommendations:")
for movie_id, score in top_recs_item_based:
    print(f"Movie {movie_id} → predicted rating: {score:.2f}")


# Model Based CF

## Explicit feedback - SVD - SGD

In [19]:
from surprise import Reader, SVD, Dataset, accuracy
from surprise.model_selection import GridSearchCV, train_test_split, cross_validate, LeaveOneOut

In [20]:
reader = Reader(rating_scale=(0.5, 5))
reader

In [21]:
data = Dataset.load_from_df(filtered[['userId',
                                       'movieId',
                                       'rating']], reader)


In [22]:
data.df.head()

,userId,movieId,rating
5400,54,1,4.0
5401,54,2,3.0
5402,54,5,3.0
5403,54,6,3.0
5404,54,10,4.0


### train_test_split

In [23]:
trainset, testset = train_test_split(data, test_size=.25)
trainset.n_users

5000

In [24]:
svd_model = SVD()
svd_model.fit(trainset)


In [25]:
predictions = svd_model.test(testset)
accuracy.rmse(predictions)


RMSE: 0.7235


0.7234757972509446

In [26]:
svd_model.predict(uid=1.0, iid=541, r_ui=4.0, verbose=True)


user: 1.0        item: 541        r_ui = 4.00   est = 3.97   {'was_impossible': False}


Prediction(uid=1.0, iid=541, r_ui=4.0, est=3.9745673806047335, details={'was_impossible': False})

In [27]:
ratings[(ratings["userId"] == 1) & (ratings["movieId"] == 541)]


,userId,movieId,rating,timestamp
15,1,541,4.0,2005-04-02 23:30:03


### Get Top N recommendations for all users

In [28]:
from collections import defaultdict

from surprise import Dataset, SVD


def get_top_n(predictions, n=10):
    """Return the top-N recommendation for each user from a set of predictions.

    Args:
        predictions(list of Prediction objects): The list of predictions, as
            returned by the test method of an algorithm.
        n(int): The number of recommendation to output for each user. Default
            is 10.

    Returns:
    A dict where keys are user (raw) ids and values are lists of tuples:
        [(raw item id, rating estimation), ...] of size n.
    """

    # First map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n

In [29]:
top_n = get_top_n(predictions, n=5)

# Print the recommended items for each user
movie_ids = []
for uid, user_ratings in top_n.items():
        movie_ids = [iid for (iid,_) in user_ratings]
        print(uid, movie_ids)

14808 [3578, 7153, 50, 4855, 2762]
137885 [1272, 904, 912, 2366, 1228]
91349 [2019, 922, 541, 912, 1212]
121535 [1207, 1221, 1219, 908, 1204]
65387 [1219, 5060, 1080, 2010, 1965]
103701 [2959, 50, 6016, 44555, 58559]
85986 [1201, 2858, 50, 60069, 6711]
131721 [318, 3949, 1193, 3160, 1210]
60887 [5669, 296, 1407, 1307, 50]
124967 [969, 50, 1214, 4720, 2692]
107317 [1221, 4848, 1219, 908, 535]
30835 [4993, 30793, 4720, 8970, 2542]
135418 [33794, 7147, 4878, 50, 4226]
22205 [79132, 318, 3949, 81591, 6874]
104977 [5971, 2329, 1298, 480, 260]
15203 [2791, 1198, 1213, 4002, 1288]
26884 [858, 903, 5060, 1203, 3072]
87954 [920, 40815, 8368, 6874, 4367]
66952 [296, 1203, 293, 6016, 4973]
100367 [5952, 4993, 2959, 4973, 593]
103199 [1266, 3504, 527, 246, 1214]
98687 [589, 1198, 2916, 5952, 1036]
35510 [7153, 4993, 2028, 1221, 1219]
57626 [33166, 2959, 4226, 58559, 7361]
128653 [1073, 1298, 29, 1285, 1032]
23437 [1250, 7153, 260, 1201, 720]
41281 [47, 2542, 2858, 858, 1233]
18007 [541, 4979, 750,

### Evaluation for all users

In [30]:
from collections import defaultdict

from surprise import Dataset, SVD
from surprise.model_selection import KFold


def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """Return precision and recall at k metrics for each user"""

    # First map the predictions to each user.
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():

        # Sort user ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Number of recommended items in top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

        # Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in user_ratings[:k]
        )

        # Precision@K: Proportion of recommended items that are relevant
        # When n_rec_k is 0, Precision is undefined. We here set it to 0.

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

        # Recall@K: Proportion of relevant items that are recommended
        # When n_rel is 0, Recall is undefined. We here set it to 0.

        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return precisions, recalls


In [31]:
precisions, recalls = precision_recall_at_k(predictions, k=5, threshold=4)

# Precision and recall can be averaged over all users
print(f"Macro Precision@5 = {sum(prec for prec in precisions.values()) / len(precisions)}")
print(f"Macro Recall@5 = {sum(rec for rec in recalls.values()) / len(recalls)}")

Macro Precision@5 = 0.8586666666666797
Macro Recall@5 = 0.08067802236177316


### Recommendation for a user

In [32]:
data.df[(data.df["userId"] == 29084)].shape

(418, 3)

In [33]:
inner_id = trainset.to_inner_uid(29084)   # convert raw → inner
trainset.knows_user(inner_id) 

#Don’t include users who aren’t in the trainset when computing Precision@K or Recall@K.
#Use a splitting method that ensures each user has ratings in both train and test.
#Otherwise, you’re evaluating the model on a cold-start scenario it wasn’t designed to handle

True

In [34]:
u = []
for t in testset:
    if t[0] == 29084:
        u.append(t)


testset2 = trainset.build_testset()
v = []
for t in testset2:
    if t[0] == 29084:
        v.append(t)

print(len(u))
print(len(v))

#Total user enties = 418 (305 + 113)
#user present in train set - 305
#user present in test set -  113 


103
315


In [35]:
for p in predictions:
    if p.uid == 29084 and p.iid== 3409:
        print(p)

In [36]:
svd_model.predict(uid=29084, iid=3409, verbose=True)

user: 29084      item: 3409       r_ui = None   est = 3.19   {'was_impossible': False}


Prediction(uid=29084, iid=3409, r_ui=None, est=3.1905221513144006, details={'was_impossible': False})

In [37]:
movie_ids = []
for uid, user_ratings in top_n.items():
    if uid == 29084:
        movie_ids = [iid for (iid,_) in user_ratings]
        print(uid, movie_ids)

29084 [293, 4973, 593, 44191, 56367]


In [38]:
upred = pd.DataFrame(top_n[29084], columns=["movieId", "prediction"])

udf = data.df[(data.df["userId"] == 29084)].sort_values(by="rating")[::-1].reset_index()
utrue = udf[udf["movieId"].isin(movie_ids)]
utrue.merge(upred, on="movieId")                           

,index,userId,movieId,rating,prediction
0,4270881,29084,4973,5.0,4.168186
1,4270485,29084,293,5.0,4.196810
2,4270525,29084,593,5.0,4.109068
3,4271236,29084,56367,4.5,4.106574
4,4271165,29084,44191,4.0,4.107431


In [39]:
#Precision@K only cares about the quality of the top‑K list. Since all 5 were relevant, precision is perfect.
#Recall@K cares about coverage. Even though the top‑5 were all correct, they represent only a small fraction of the user’s total relevant items. That’s why recall is low.

print(precisions[29084])

# t[0] == 29084 and t[2] > 3.5 = 48 
# 5/48 = 0.104
print(recalls[29084])

1.0
0.11627906976744186


### Leave one out split 
Every user is represented in both train and test. That way, you won’t run into “user not in trainset” mismatches.

In [40]:
data.df.shape

(2353665, 3)

In [41]:
# LeaveOneOut split: each user has at least one rating in test
loo = LeaveOneOut(n_splits=1, random_state=42)

for trainset, testset in loo.split(data):
    print("Trainset size:", trainset.n_ratings)
    print("Testset size:", len(testset))

Trainset size: 2348665
Testset size: 5000


In [42]:
# Check that every user is in trainset
all_users = set(filtered['userId'].unique())
train_users = set(trainset.to_raw_uid(i) for i in range(trainset.n_users))
print("Users missing from trainset:", all_users - train_users)  # should be empty

Users missing from trainset: set()


## Implicit Feedback - SVD - Numpy

In [ ]:
# Matrix factorization via SVD
#TODO
U, sigma, Vt = np.linalg.svd(user_item, full_matrices=False)
pred_matrix = np.dot(U, np.dot(np.diag(sigma), Vt))

## Implicit Feedback - ALS

In [43]:
from scipy import sparse
import implicit

C:\Users\nedwi\AppData\Local\Programs\Python\Python312\312-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
path = os.path.abspath(os.path.join((os.getcwd()),  os.pardir, "data\\retail\\Online_Retail.xlsx")) 
df_raw = pd.read_excel(path)
print(df_raw.shape)
df_raw.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [45]:
df_raw["InvoiceNo"].nunique()

25900

In [46]:
print(f"Total users: {df_raw["CustomerID"].nunique()}")
print(f"Total items: {df_raw["StockCode"].nunique()}")

Total users: 4372
Total items: 4070


In [47]:
#count of unique Items bought per customer
df_items_per_cust = (
    df_raw.groupby(["CustomerID"]).agg({"StockCode": "nunique"}).reset_index()
)
df_items_per_cust.columns = ["CustomerID", "Count_item_cust"]
df_items_per_cust

,CustomerID,Count_item_cust
0,12346.0,1
1,12347.0,103
2,12348.0,22
3,12349.0,73
4,12350.0,17
...,...,...
4367,18280.0,10
4368,18281.0,7
4369,18282.0,12
4370,18283.0,263


In [48]:
# Setting of Threshold
item_in_cust_threshold = 6

# Filtering Results
mask = df_items_per_cust["Count_item_cust"] >= item_in_cust_threshold
valid_cust = set(df_items_per_cust.loc[mask, "CustomerID"].tolist())

df_filter_cust = df_raw[df_raw["CustomerID"].isin(valid_cust)].copy()
invoiceno_filter_cust = set(df_filter_cust["InvoiceNo"].tolist())

In [49]:
#count of unique customers who bought that item
df_custs_per_item = (
    df_raw.groupby(["StockCode"]).agg({"CustomerID": "nunique"}).reset_index()
)
df_custs_per_item.columns = ["StockCode", "Count_cust_item"]
df_custs_per_item["Count_cust_item"].value_counts()

Count_cust_item
0      386
1      189
2      145
3      116
6       83
      ... 
252      1
392      1
858      1
276      1
379      1
Name: count, Length: 380, dtype: int64

In [50]:
# Set threshold
cust_in_item_threshold = 6
#Filter Results
mask = df_custs_per_item["Count_cust_item"] >= cust_in_item_threshold
valid_stockcode = set(df_custs_per_item.loc[mask, "StockCode"].tolist())

df_filter_item = df_raw[df_raw["StockCode"].isin(valid_stockcode)].copy()
invoiceno_filter_item = set(df_filter_item["InvoiceNo"].tolist())

In [51]:
invoiceno_intersect = set.intersection(invoiceno_filter_item, invoiceno_filter_cust)
df_filter_cust_item = df_raw[df_raw["InvoiceNo"].isin(invoiceno_intersect)].copy()

In [52]:
df_filter_cust_item.shape

(405521, 8)

In [53]:
#Generating Customer and Item IDs
unique_customers = df_filter_cust_item["CustomerID"].unique()
cust_ids = dict(
    zip(unique_customers, np.arange(unique_customers.shape[0], dtype=np.int32))
)
print(len(cust_ids))

unique_items = df_filter_cust_item["StockCode"].unique()
item_ids = dict(zip(unique_items, np.arange(unique_items.shape[0], dtype=np.int32)))
print(len(item_ids))


4017
3671


In [54]:
df_filter_cust_item["cust_id"] = df_filter_cust_item["CustomerID"].apply(
    lambda i: cust_ids[i]
)
df_filter_cust_item["item_id"] = df_filter_cust_item["StockCode"].apply(
    lambda i: item_ids[i]
)

df_filter_cust_item.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,cust_id,item_id
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,1
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,2
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,3
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,4


In [55]:
df_cust_item_qty = (
    df_filter_cust_item.groupby(["cust_id", "item_id"])
    .agg({"Quantity": "sum"})
    .reset_index()
)
df_cust_item_qty

,cust_id,item_id,Quantity
0,0,0,122
1,0,1,122
2,0,2,108
3,0,3,110
4,0,4,104
...,...,...,...
266591,4016,3219,6
266592,4016,3271,12
266593,4016,3320,25
266594,4016,3452,4


In [56]:
# Create Sparse Matrix
sparse_customer_item = sparse.csr_matrix(
    (
        df_cust_item_qty["Quantity"].astype(float),
        (df_cust_item_qty["cust_id"], df_cust_item_qty["item_id"]), #shape
    ) 
)
sparse_customer_item

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 266596 stored elements and shape (4017, 3671)>

In [57]:
# Initialize an Alternating Least Squares model
model = implicit.als.AlternatingLeastSquares(factors=50)
# Train on user-item sparse matrix
model.fit(sparse_customer_item)



C:\Users\nedwi\AppData\Local\Programs\Python\Python312\312-env\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:01<00:00,  7.51it/s]


In [58]:
print(model.user_factors.shape)
print(model.item_factors.shape)

(4017, 50)
(3671, 50)


In [59]:
# Get recommendations
cust_id = 0
ids, scores = model.recommend(
    cust_id, sparse_customer_item[cust_id], N=5, filter_already_liked_items=False
)

len(scores)

5

In [60]:
df_item_desc = df_filter_cust_item.groupby("StockCode")[["Description", "item_id"]].first().reset_index()
print(df_item_desc.shape)

(3671, 3)


In [61]:
df_item_desc.head()

,StockCode,Description,item_id
0,10002,INFLATABLE POLITICAL GLOBE,31
1,10080,GROOVY CACTUS INFLATABLE,2761
2,10120,DOGGY RUBBER,1334
3,10125,MINI FUNKY DESIGN TAPES,502
4,10133,COLOURING PENCILS BROWN TUBE,460


In [62]:
list_stockcode = df_item_desc[df_item_desc["item_id"].isin(ids)]["StockCode"].tolist()
list_desc = df_item_desc[df_item_desc["item_id"].isin(ids)]["Description"].tolist()
df_recommendations = pd.DataFrame(
    {
        "Stockcode": list_stockcode,
        "Description": list_desc,
        "score": scores,
        "already_liked": np.in1d(ids, sparse_customer_item[cust_id].indices),
    }
)
df_recommendations

,Stockcode,Description,score,already_liked
0,21733,RED HANGING HEART T-LIGHT HOLDER,1.259391,False
1,22111,SCOTTIE DOG HOT WATER BOTTLE,1.155702,False
2,22113,GREY HEART HOT WATER BOTTLE,1.147058,False
3,22834,HAND WARMER BABUSHKA DESIGN,1.109052,False
4,71477,COLOUR GLASS. STAR T-LIGHT HOLDER,1.052645,True


In [63]:

#Finding similar items 
#Given a particular item, a list of closest items can be generated

ref_item_id = df_filter_cust_item["item_id"].unique()
item_arr, score_arr = model.similar_items(ref_item_id, N=10)

In [64]:
df_item_temp = pd.DataFrame(item_arr)
df_item_temp["Ref Item ID"] = ref_item_id
df_item_rank = pd.melt(
    df_item_temp,
    id_vars=["Ref Item ID"],
    var_name="Item Rank",
    value_name="Related Item ID",
)
df_score_temp = pd.DataFrame(score_arr)
df_score_temp["Ref Item ID"] = ref_item_id
df_score_rank = pd.melt(
    df_score_temp, id_vars=["Ref Item ID"], var_name="Item Rank", value_name="Score"
)
df_item_score = df_item_rank.merge(
    df_score_rank, how="inner", on=["Ref Item ID", "Item Rank"]
)

In [65]:
df_item_score

,Ref Item ID,Item Rank,Related Item ID,Score
0,0,0,0,1.000000
1,1,0,1,1.000000
2,2,0,2,1.000000
3,3,0,3,1.000000
4,4,0,4,1.000000
...,...,...,...,...
36705,3666,9,3173,0.880004
36706,3667,9,3173,0.880147
36707,3668,9,3173,0.880212
36708,3669,9,3173,0.880144


In [66]:
# Time based split

In [67]:
import pandas as pd

# Example: user activity log with monthly churn labels
data = pd.DataFrame({
    "user_id": [1,1,1,2,2,2],
    "month": pd.to_datetime(["2024-12","2025-01","2025-02",
                              "2024-12","2025-01","2025-02"]),
    "logins": [10, 8, 5, 7, 6, 2],
    "churn_label": [0, 0, 1, 0, 1, 1]  # 1 = churn in that month
})

# -----------------------------
# 1. Build features snapshot
# -----------------------------
# Features are based on past month activity
X = data[["user_id","month","logins"]]

# -----------------------------
# 2. Align labels with horizon
# -----------------------------
# For 1-month ahead churn prediction:
# Label = churn in the NEXT month
data["label_next_month"] = data.groupby("user_id")["churn_label"].shift(-1)

# Now each row has features from month t and label from month t+1
aligned = data.dropna(subset=["label_next_month"])

In [68]:
data

,user_id,month,logins,churn_label,label_next_month
0,1,2024-12-01,10,0,0.0
1,1,2025-01-01,8,0,1.0
2,1,2025-02-01,5,1,NaN
3,2,2024-12-01,7,0,1.0
4,2,2025-01-01,6,1,1.0
5,2,2025-02-01,2,1,NaN


In [69]:
aligned

,user_id,month,logins,churn_label,label_next_month
0,1,2024-12-01,10,0,0.0
1,1,2025-01-01,8,0,1.0
3,2,2024-12-01,7,0,1.0
4,2,2025-01-01,6,1,1.0


In [70]:
train = aligned[aligned["month"] <= "2024-12"]
test  = aligned[(aligned["month"] >= "2025-01") & (aligned["month"] <= "2025-02")]

X_train, y_train = train[["logins"]], train["label_next_month"]
X_test, y_test   = test[["logins"]], test["label_next_month"]


In [71]:
train

,user_id,month,logins,churn_label,label_next_month
0,1,2024-12-01,10,0,0.0
3,2,2024-12-01,7,0,1.0


In [72]:
test

,user_id,month,logins,churn_label,label_next_month
1,1,2025-01-01,8,0,1.0
4,2,2025-01-01,6,1,1.0
